# Tutorial 4, Q2a: factory A on its own

Release after the practical.

**The problem.** Factory A makes standard ($s$, `x[0]`) and deluxe ($d$, `x[1]`) units with profits £10
and £15. Maximise $10s + 15d$ subject to grinding $4s + 2d \le 80$ hours, polishing
$2s + 5d \le 60$ hours, raw material $4s + 4d \le 75$ kg (factory A's share of the 120 kg),
$s, d \ge 0$ and whole numbers (the answers use whole units; real values are also fine).

**How to run it.** Run the first code cell first (it installs Pyomo and the HiGHS solver; the first time takes about a minute), then *Run → Run All Cells*, or run the cells one by one while you read.

**Expected result.** 10 standard and 8 deluxe, profit **£220**; unused hours: grinding 24, polishing 0 (and 3 kg of raw material left).

*Original script: `5 Source Files/Python/tutorial_q2a_factoryA.py`. Runs in the browser with HiGHS instead of GLPK; the model and answers are unchanged.*

In [ ]:
# Run this cell first.
# In the browser (JupyterLite) it installs the extra packages into this tab;
# in Colab or Anaconda it installs them with pip (already installed = no change).
%pip install pyomo highspy

import sys
if sys.platform == "emscripten":  # only in the browser (JupyterLite): adapt Pyomo to it
    import contextlib, pyomo.common.tee, pyomo.common.formatting as _fmt
    # 1) browsers have no threads: skip Pyomo's threaded capture of solver output
    pyomo.common.tee.capture_output = lambda *args, **kwargs: contextlib.nullcontext()
    for _m in ("pyomo.contrib.appsi.solvers.highs", "pyomo.contrib.solver.solvers.highs"):
        if _m in sys.modules:
            sys.modules[_m].capture_output = pyomo.common.tee.capture_output
    # 2) the notebook's output returns None from write(); model.pprint()/display() need a number
    if not getattr(_fmt.StreamIndenter, "_browser_patched", False):
        class _Counted:
            def __init__(self, s): self._s = s
            def write(self, data): self._s.write(data); return len(data)
            def __getattr__(self, name): return getattr(self._s, name)
        def _init(self, ostream, indent=" " * 4, _orig=_fmt.StreamIndenter.__init__):
            _orig(self, ostream, indent)
            if not isinstance(self.target_os, _Counted):
                object.__setattr__(self, "target_os", _Counted(self.target_os))
        _fmt.StreamIndenter.__init__ = _init
        _fmt.StreamIndenter._browser_patched = True

Load Pyomo's modelling tools.

In [ ]:
from pyomo.environ import *

A *concrete model* is the container for the problem.

In [ ]:
# Create a Concrete Model
model = ConcreteModel()

Two whole-number variables: `x[0]` = standard units $s$, `x[1]` = deluxe units $d$.

In [ ]:
# Integer decision variables: x[0] = standard units (s), x[1] = deluxe units (d), both >= 0
model.x = Var(range(2), domain=NonNegativeIntegers)

Profit per unit: £10 standard, £15 deluxe.

In [ ]:
# Profit per unit (£): standard, deluxe
profit = [10, 15]

Resource use per unit, from the table on the sheet: grinding and polishing hours at factory A, and 4 kg of raw material per unit.

In [ ]:
# Resource use per unit [standard, deluxe]: grinding hours, polishing hours, raw material kg
grind = [4, 2]
polish = [2, 5]
raw = [4, 4]

The objective: total profit $10s + 15d$, maximised.

In [ ]:
# Objective: maximise total profit 10s + 15d
model.obj = Objective(expr=sum(profit[i] * model.x[i] for i in model.x), sense=maximize)

One constraint per limited resource.

In [ ]:
# Define the inequality constraints
model.grinding_cap = Constraint(expr=sum(grind[i] * model.x[i] for i in model.x) <= 80)  # grinding: 4s + 2d <= 80 hours
model.polishing_cap = Constraint(expr=sum(polish[i] * model.x[i] for i in model.x) <= 60)  # polishing: 2s + 5d <= 60 hours
model.raw_cap = Constraint(expr=sum(raw[i] * model.x[i] for i in model.x) <= 75)  # raw material: 4s + 4d <= 75 kg

Solve with HiGHS; `tee=True` prints the solver's log. Near the end it should say the model status is *Optimal*.

In [ ]:
# Solve the optimization problem
solver = SolverFactory('appsi_highs')  # HiGHS runs in the browser (the original script used GLPK)
solver.solve(model, tee=True)

`pprint()` prints the whole model with the data filled in: check that it matches the formulation above.

In [ ]:
# Print the model
model.pprint()

`display()` prints the values: the number of each product, the profit, and how much of each resource is used next to its limit.

In [ ]:
# Print the results
model.display()

Q2a ii asks how many hours are left. Put the solution back into each constraint and subtract the amount used from the capacity. The code calls this *surplus*; in the lecture's terms (the *Slack Variables* slides) unused capacity in a $\le$ constraint is the **slack**. A slack of 0 means the constraint is **binding**.

In [ ]:
# Slack (unused capacity) = capacity - amount used at the optimal solution
surplus_grind = 80 - sum(grind[i] * value(model.x[i]) for i in model.x)
surplus_polish = 60 - sum(polish[i] * value(model.x[i]) for i in model.x)
surplus_raw = 75 - sum(raw[i] * value(model.x[i]) for i in model.x)

print("Surplus in grinding capacity:", surplus_grind)
print("Surplus in polishing capacity:", surplus_polish)
print("Surplus in raw material capacity:", surplus_raw)

## What do you notice?

Which resource limits the plan? And what would the profit be if fractions of a unit were allowed (change
`NonNegativeIntegers` to `NonNegativeReals` and run all cells)?

<details><summary>Answer</summary>

Polishing is binding (slack 0): $2 \times 10 + 5 \times 8 = 60$ hours. Grinding has 24 hours spare and 3 kg of
raw material is unused.

With fractional units the LP optimum is 11.25 standard and 7.5 deluxe, profit £225: £5 more, but not a whole
number of units.
</details>